# Molab Processing

## Syncing with keypoints

### Sync full data and save

In [4]:
import pandas as pd
import numpy as np
import os

MOLAB_FS = 100
VIDEO_FPS = 50

mapping = [
    ('0028', 'Fp1_SB', '2c'),
    ('0030', 'Fp1_SB', '2e'),
    ('0045', 'Fp2_CF', '2c'),
    ('0047', 'Fp2_CF', '2e'),
    ('0059', 'Fp3_JN', '2c'),
    ('0061', 'Fp3_JN', '2e'),
    ('0074', 'Fp4_AL', '2c'),
    ('0076', 'Fp4_AL', '2e'),
    ('0091', 'Fp5_WL', '2c'),
    ('0093', 'Fp5_WL', '2e'),
]

beep_times = {
    '0028': 9.14,
    '0030': 5.38,
    '0045': 5.88,
    '0047': 6.12,
    '0059': 5.70,
    '0061': 6.10,
    '0074': 5.42,
    '0076': 5.54,
    '0091': 5.52,
    '0093': 5.00,
}

corrections = {
    '0028': 0.52,
}

out_dir = '../data/processed/molab'
os.makedirs(out_dir, exist_ok=True)

for vid, fp, mid in mapping:
    beep_time = beep_times[vid]
    correction = corrections.get(vid, 0)
    offset = beep_time - 4 + correction

    joint = pd.read_csv(f'../data/original/MoLab/{fp}/{mid}_JointAngle.csv', sep=',', decimal=',')
    acc   = pd.read_csv(f'../data/original/MoLab/{fp}/{mid}_SegmentAcc.csv', sep=',', decimal=',')
    gyro  = pd.read_csv(f'../data/original/MoLab/{fp}/{mid}_SegmentGyro.csv', sep=',', decimal=',')
    quat  = pd.read_csv(f'../data/original/MoLab/{fp}/{mid}_SegmentQuaternions_Base.csv', sep=',', decimal=',')

    molab_frame = joint['Index'].astype(int)
    molab_time  = (molab_frame / MOLAB_FS).round(2)
    offset_in_molab_samples = int(round(offset * MOLAB_FS))
    video_frame = (molab_frame + offset_in_molab_samples) // 2
    video_time  = (video_frame / VIDEO_FPS).round(2)

    data = pd.concat([
        joint.drop(columns=['Index']),
        acc.drop(columns=['Index']),
        gyro.drop(columns=['Index']),
        quat.drop(columns=['Index']),
    ], axis=1)

    data.insert(0, 'molab_frame', molab_frame.values)
    data.insert(1, 'molab_time', molab_time.values)
    data.insert(2, 'video_frame', video_frame.values)
    data.insert(3, 'video_time', video_time.values)

    out_path = f'{out_dir}/{vid}_molab.csv'
    data.to_csv(out_path, index=False)
    print(f'Saved {out_path}: {len(data)} rows, {len(data.columns)} columns')


Saved ../data/processed/molab/0028_molab.csv: 1798 rows, 95 columns
Saved ../data/processed/molab/0030_molab.csv: 4026 rows, 95 columns
Saved ../data/processed/molab/0045_molab.csv: 1990 rows, 95 columns
Saved ../data/processed/molab/0047_molab.csv: 3549 rows, 95 columns
Saved ../data/processed/molab/0059_molab.csv: 2657 rows, 95 columns
Saved ../data/processed/molab/0061_molab.csv: 3850 rows, 95 columns
Saved ../data/processed/molab/0074_molab.csv: 1843 rows, 95 columns
Saved ../data/processed/molab/0076_molab.csv: 3665 rows, 95 columns
Saved ../data/processed/molab/0091_molab.csv: 1466 rows, 95 columns
Saved ../data/processed/molab/0093_molab.csv: 3709 rows, 95 columns


### Save synced + cropped data

In [5]:
import glob

for kp_cropped_dir, out_dir_cropped, label in [
    ('../data/processed/keypoints_cropped',      '../data/processed/molab_cropped',      'sit-to-stand'),
    ('../data/processed/keypoints_cropped_down', '../data/processed/molab_cropped_down', 'stand-to-sit'),
]:
    os.makedirs(out_dir_cropped, exist_ok=True)
    print(f'\n=== {label} ===')
    for vid, fp, mid in mapping:
        kp_files = glob.glob(f'{kp_cropped_dir}/**/*_{vid}_*.csv', recursive=True)
        if not kp_files:
            print(f'{vid}: no cropped keypoints file found, skipping')
            continue
        kp = pd.read_csv(kp_files[0])
        frame_min = kp['frame'].min()
        frame_max = kp['frame'].max()

        molab = pd.read_csv(f'../data/processed/molab/{vid}_molab.csv')
        cropped = molab[(molab['video_frame'] >= frame_min) & (molab['video_frame'] <= frame_max)].copy()

        out_path = f'{out_dir_cropped}/{vid}_molab.csv'
        cropped.to_csv(out_path, index=False)
        print(f'{vid}: frames {frame_min}-{frame_max} -> {len(cropped)} rows saved to {out_path}')


=== sit-to-stand ===
0028: frames 495-588 -> 188 rows saved to ../data/processed/molab_cropped/0028_molab.csv
0030: frames 290-381 -> 184 rows saved to ../data/processed/molab_cropped/0030_molab.csv
0045: frames 327-399 -> 146 rows saved to ../data/processed/molab_cropped/0045_molab.csv
0047: frames 352-445 -> 188 rows saved to ../data/processed/molab_cropped/0047_molab.csv
0059: frames 333-455 -> 246 rows saved to ../data/processed/molab_cropped/0059_molab.csv
0061: frames 300-450 -> 302 rows saved to ../data/processed/molab_cropped/0061_molab.csv
0074: frames 304-363 -> 120 rows saved to ../data/processed/molab_cropped/0074_molab.csv
0076: frames 250-400 -> 302 rows saved to ../data/processed/molab_cropped/0076_molab.csv
0091: frames 320-380 -> 122 rows saved to ../data/processed/molab_cropped/0091_molab.csv
0093: frames 293-383 -> 182 rows saved to ../data/processed/molab_cropped/0093_molab.csv

=== stand-to-sit ===
0028: frames 594-643 -> 100 rows saved to ../data/processed/molab_

### Plot molab vs keypoint to compare timing

In [ ]:
import matplotlib.pyplot as plt
import glob
import os

MOLAB_FULL_DIR = '../data/processed/molab'
ANGLE_PLOTS_DIR = '../data/processed/angle_plots'
VIDEO_ASPECT = 1920 / 1080

PLOT_JOINTS = [
    ('L_knee_X',  'Left Knee'),
    ('R_knee_X',  'Right Knee'),
    ('L_hip_X',   'Left Hip'),
    ('R_hip_X',   'Right Hip'),
    ('L_ankle_X', 'Left Ankle'),
    ('R_ankle_X', 'Right Ankle'),
    ('pelvis_X',  'Pelvis'),
]

offsets = {}
for vid, fp, mid in mapping:
    full = pd.read_csv(f'{MOLAB_FULL_DIR}/{vid}_molab.csv')
    pre  = full[full['molab_frame'] < 41]
    offsets[vid] = {j: pre[f'JointAngle/{j}'].mean() for j, _ in PLOT_JOINTS}


def joint_angle_2d(ax, ay, bx, by, cx, cy):
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    dot  = v1x * v2x + v1y * v2y
    norm = np.sqrt(v1x**2 + v1y**2) * np.sqrt(v2x**2 + v2y**2) + 1e-8
    return 180 - np.degrees(np.arccos(np.clip(dot / norm, -1, 1)))


def segment_angle_2d(ax, ay, bx, by, cx, cy, dx, dy):
    v1x, v1y = bx - ax, by - ay
    v2x, v2y = dx - cx, dy - cy
    dot  = v1x * v2x + v1y * v2y
    norm = np.sqrt(v1x**2 + v1y**2) * np.sqrt(v2x**2 + v2y**2) + 1e-8
    return 180 - np.degrees(np.arccos(np.clip(dot / norm, -1, 1)))


def trunk_angle_2d(kp):
    mid_hip_x = (kp['left_hip_x'] + kp['right_hip_x']) / 2
    mid_hip_y = (kp['left_hip_y'] + kp['right_hip_y']) / 2
    mid_sh_x  = (kp['left_shoulder_x'] + kp['right_shoulder_x']) / 2
    mid_sh_y  = (kp['left_shoulder_y'] + kp['right_shoulder_y']) / 2
    dx = (mid_sh_x - mid_hip_x).values
    dy = (mid_sh_y - mid_hip_y).values
    return -np.degrees(np.arctan2(dx, -dy))


def compute_kp_angles(kp):
    kp = kp.copy()
    x_cols = [c for c in kp.columns if c.endswith('_x')]
    kp[x_cols] *= VIDEO_ASPECT

    angles = {}
    for side, s in [('L', 'left'), ('R', 'right')]:
        angles[f'{side}_knee_X'] = joint_angle_2d(
            kp[f'{s}_hip_x'].values,    kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,   kp[f'{s}_knee_y'].values,
            kp[f'{s}_ankle_x'].values,  kp[f'{s}_ankle_y'].values,
        )
        angles[f'{side}_hip_X'] = joint_angle_2d(
            kp[f'{s}_shoulder_x'].values, kp[f'{s}_shoulder_y'].values,
            kp[f'{s}_hip_x'].values,      kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,     kp[f'{s}_knee_y'].values,
        )
        angles[f'{side}_ankle_X'] = segment_angle_2d(
            kp[f'{s}_heel_x'].values,       kp[f'{s}_heel_y'].values,
            kp[f'{s}_foot_index_x'].values, kp[f'{s}_foot_index_y'].values,
            kp[f'{s}_ankle_x'].values,      kp[f'{s}_ankle_y'].values,
            kp[f'{s}_knee_x'].values,       kp[f'{s}_knee_y'].values,
        )
    angles['pelvis_X'] = trunk_angle_2d(kp)
    return angles


def compute_kp_angles_world(kp):
    kp = kp.copy()

    angles = {}
    for side, s in [('L', 'left'), ('R', 'right')]:
        angles[f'{side}_knee_X'] = joint_angle_2d(
            kp[f'{s}_hip_x'].values,    kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,   kp[f'{s}_knee_y'].values,
            kp[f'{s}_ankle_x'].values,  kp[f'{s}_ankle_y'].values,
        )
        angles[f'{side}_hip_X'] = joint_angle_2d(
            kp[f'{s}_shoulder_x'].values, kp[f'{s}_shoulder_y'].values,
            kp[f'{s}_hip_x'].values,      kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,     kp[f'{s}_knee_y'].values,
        )
        angles[f'{side}_ankle_X'] = segment_angle_2d(
            kp[f'{s}_heel_x'].values,       kp[f'{s}_heel_y'].values,
            kp[f'{s}_foot_index_x'].values, kp[f'{s}_foot_index_y'].values,
            kp[f'{s}_ankle_x'].values,      kp[f'{s}_ankle_y'].values,
            kp[f'{s}_knee_x'].values,       kp[f'{s}_knee_y'].values,
        )
    mid_hip_x = (kp['left_hip_x'] + kp['right_hip_x']) / 2
    mid_hip_y = (kp['left_hip_y'] + kp['right_hip_y']) / 2
    mid_sh_x  = (kp['left_shoulder_x'] + kp['right_shoulder_x']) / 2
    mid_sh_y  = (kp['left_shoulder_y'] + kp['right_shoulder_y']) / 2
    dx = (mid_sh_x - mid_hip_x).values
    dy = (mid_sh_y - mid_hip_y).values
    angles['pelvis_X'] = np.degrees(np.arctan2(dx, dy))
    return angles


def plot_molab_angles(molab_dir, kp_dir, kp_pattern, title, out_dir, label,
                      world_kp_dir=None, world_kp_pattern=None):
    os.makedirs(out_dir, exist_ok=True)
    for vid, fp, mid in mapping:
        molab  = pd.read_csv(f'{molab_dir}/{vid}_molab.csv')
        offset = beep_times[vid] - 4 + corrections.get(vid, 0)
        t    = molab['molab_time'].values + offset
        mask = molab['molab_frame'].values >= 41

        kp_files  = glob.glob(f'{kp_dir}/{kp_pattern.format(vid=vid)}')
        kp_angles = None
        kp_time   = None
        if kp_files:
            kp        = pd.read_csv(kp_files[0])
            kp_time   = kp['frame'].values / VIDEO_FPS
            kp_angles = compute_kp_angles(kp)

        world_files  = glob.glob(f'{world_kp_dir}/{world_kp_pattern.format(vid=vid)}') if world_kp_dir else []
        world_angles = None
        world_time   = None
        if world_files:
            kp_w         = pd.read_csv(world_files[0])
            world_time   = kp_w['frame'].values / VIDEO_FPS
            world_angles = compute_kp_angles_world(kp_w)

        fig, axes = plt.subplots(4, 2, figsize=(14, 16))
        fig.suptitle(f'{title} {vid}: {fp}', fontsize=13, fontweight='bold')

        for row in range(3):
            axes[row, 1].sharey(axes[row, 0])

        for i, (joint, jlabel) in enumerate(PLOT_JOINTS):
            ax  = axes.flat[i]
            raw = molab[f'JointAngle/{joint}'].values.copy()
            raw[mask] += offsets[vid][joint]
            ax.plot(t, raw, linewidth=1.2, color='steelblue', label='MoLab')

            if kp_angles is not None and joint in kp_angles:
                ax.plot(kp_time, kp_angles[joint], linewidth=1.2,
                        color='tomato', alpha=0.8, label='SAT norm')

            if world_angles is not None and joint in world_angles:
                ax.plot(world_time, world_angles[joint], linewidth=1.2,
                        color='seagreen', alpha=0.8, label='SAT world')

            ax.set_title(jlabel)
            ax.set_xlabel('Video time (s)')
            ax.set_ylabel('Angle (°)')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7)

        axes.flat[-1].set_visible(False)
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{vid}_{fp}_{label}.png', dpi=150, bbox_inches='tight')
        plt.show()


plot_molab_angles(
    '../data/processed/molab',
    '../data/processed/keypoints/mediapipe_norm',
    '*_{vid}_D_mediapipe_norm.csv',
    'MoLab vs SAT Full',
    f'{ANGLE_PLOTS_DIR}/full',
    'full',
)
plot_molab_angles(
    '../data/processed/molab_cropped',
    '../data/processed/keypoints_combined',
    'DJI_*_{vid}_D.csv',
    'MoLab vs SAT Sit-to-stand',
    f'{ANGLE_PLOTS_DIR}/sit_to_stand',
    'sit_to_stand',
    world_kp_dir='../data/processed/keypoints_combined_world',
    world_kp_pattern='DJI_*_{vid}_D.csv',
)
plot_molab_angles(
    '../data/processed/molab_cropped_down',
    '../data/processed/keypoints_combined_down',
    'DJI_*_{vid}_D.csv',
    'MoLab vs SAT Stand-to-sit',
    f'{ANGLE_PLOTS_DIR}/stand_to_sit',
    'stand_to_sit',
    world_kp_dir='../data/processed/keypoints_combined_world_down',
    world_kp_pattern='DJI_*_{vid}_D.csv',
)

In [ ]:
import matplotlib.pyplot as plt
import glob
import os

KNEE_ANGLE_PLOTS_DIR = '../data/processed/knee_angle_plots'
os.makedirs(KNEE_ANGLE_PLOTS_DIR, exist_ok=True)

knee_offsets = {}
for vid, fp, mid in mapping:
    full = pd.read_csv(f'{MOLAB_FULL_DIR}/{vid}_molab.csv')
    pre  = full[full['molab_frame'] < 41]
    knee_offsets[vid] = (pre['JointAngle/L_knee_X'].mean() + pre['JointAngle/R_knee_X'].mean()) / 2


def knee_angle_2d(hip_x, hip_y, knee_x, knee_y, ankle_x, ankle_y, scale_x=1.0):
    hip_x, knee_x, ankle_x = hip_x * scale_x, knee_x * scale_x, ankle_x * scale_x
    v1 = np.stack([hip_x - knee_x, hip_y - knee_y], axis=1)
    v2 = np.stack([ankle_x - knee_x, ankle_y - knee_y], axis=1)
    cos_angle = np.sum(v1 * v2, axis=1) / (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1) + 1e-8)
    return 180 - np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))


def plot_molab_vs_kp(molab_dir, kp_dir, title, out_name,
                     world_kp_dir=None, tick_step=None):
    fig, axes = plt.subplots(5, 2, figsize=(18, 20))
    axes = axes.flatten()

    for i, (vid, fp, mid) in enumerate(mapping):
        ax = axes[i]

        molab  = pd.read_csv(f'{molab_dir}/{vid}_molab.csv')
        offset = beep_times[vid] - 4 + corrections.get(vid, 0)
        t      = molab['molab_time'].values + offset
        mask   = molab['molab_frame'].values >= 41

        molab_knee_raw = (molab['JointAngle/L_knee_X'].values + molab['JointAngle/R_knee_X'].values) / 2
        molab_knee = molab_knee_raw.copy()
        molab_knee[mask] += knee_offsets[vid]

        kp_files = glob.glob(f'{kp_dir}/*_{vid}_D_mediapipe_norm.csv')
        if kp_files:
            kp = pd.read_csv(kp_files[0])
            kp_time = kp['frame'].values / VIDEO_FPS
            left_angle = knee_angle_2d(
                kp['left_hip_x'].values, kp['left_hip_y'].values,
                kp['left_knee_x'].values, kp['left_knee_y'].values,
                kp['left_ankle_x'].values, kp['left_ankle_y'].values,
                scale_x=VIDEO_ASPECT,
            )
            right_angle = knee_angle_2d(
                kp['right_hip_x'].values, kp['right_hip_y'].values,
                kp['right_knee_x'].values, kp['right_knee_y'].values,
                kp['right_ankle_x'].values, kp['right_ankle_y'].values,
                scale_x=VIDEO_ASPECT,
            )
            kp_knee = (left_angle + right_angle) / 2
            ax.plot(kp_time, kp_knee, label='SAT norm', color='tomato', linewidth=1, alpha=0.8)

        if world_kp_dir:
            world_files = glob.glob(f'{world_kp_dir}/DJI_*_{vid}_D.csv')
            if world_files:
                kp_w = pd.read_csv(world_files[0])
                world_time = kp_w['frame'].values / VIDEO_FPS
                left_w = knee_angle_2d(
                    kp_w['left_hip_x'].values, kp_w['left_hip_y'].values,
                    kp_w['left_knee_x'].values, kp_w['left_knee_y'].values,
                    kp_w['left_ankle_x'].values, kp_w['left_ankle_y'].values,
                )
                right_w = knee_angle_2d(
                    kp_w['right_hip_x'].values, kp_w['right_hip_y'].values,
                    kp_w['right_knee_x'].values, kp_w['right_knee_y'].values,
                    kp_w['right_ankle_x'].values, kp_w['right_ankle_y'].values,
                )
                world_knee = (left_w + right_w) / 2
                ax.plot(world_time, world_knee, label='SAT world', color='seagreen', linewidth=1, alpha=0.8)

        ax.plot(t, molab_knee, label='MoLab (avg knee)', color='steelblue', linewidth=1)
        ax.set_title(f'{vid}: {fp} {mid}')
        ax.set_xlabel('Video time (s)')
        ax.set_ylabel('Knee angle (degrees)')
        ax.legend(fontsize=8)

        if tick_step is not None:
            all_times = t
            ax.set_xticks(np.arange(
                np.floor(all_times.min() / tick_step) * tick_step,
                all_times.max() + tick_step,
                tick_step,
            ))
        ax.tick_params(axis='x', labelsize=7)
        ax.grid(True, which='major', alpha=0.4)

    plt.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{KNEE_ANGLE_PLOTS_DIR}/{out_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_molab_vs_kp(
    '../data/processed/molab',
    '../data/processed/keypoints/mediapipe_norm',
    'MoLab vs Mediapipe: full data',
    'full',
    tick_step=1,
)
plot_molab_vs_kp(
    '../data/processed/molab_cropped',
    '../data/processed/keypoints_cropped/mediapipe_norm',
    'MoLab vs Mediapipe: sit-to-stand cropped',
    'sit_to_stand',
    world_kp_dir='../data/processed/keypoints_combined_world',
)
plot_molab_vs_kp(
    '../data/processed/molab_cropped_down',
    '../data/processed/keypoints_cropped_down/mediapipe_norm',
    'MoLab vs Mediapipe: stand-to-sit cropped',
    'stand_to_sit',
    world_kp_dir='../data/processed/keypoints_combined_world_down',
)

## Derive X and Y coordinates from molab data

### Compute x and y positions for full and cropped molab data

In [8]:
participants = {
    'Fp1_SB': (167, 39, 'F'),
    'Fp2_CF': (168, 38, 'F'),
    'Fp3_JN': (178, 43, 'M'),
    'Fp4_AL': (168, 38, 'F'),
    'Fp5_WL': (169, 41, 'M'),
}

RATIOS = {
    'M': {'thigh': 0.232, 'shank': 0.247},
    'F': {'thigh': 0.249, 'shank': 0.250},
}

EU_TO_FOOT_LENGTH_MM = {
    38: 240.1,
    39: 246.8,
    41: 260.1,
    43: 273.5,
}

SHANK_LONG_AXIS = np.array([0.0, 1.0, 0.0])
THIGH_LONG_AXIS = np.array([0.0, 0.0, 1.0])
FOOT_LONG_AXIS  = np.array([0.0, 0.0, 1.0])

def rotate_vecs(q, v):
    w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    vx, vy, vz = v[0], v[1], v[2]
    tx = 2 * (y * vz - z * vy)
    ty = 2 * (z * vx - x * vz)
    tz = 2 * (x * vy - y * vx)
    return np.stack([
        vx + w * tx + y * tz - z * ty,
        vy + w * ty + z * tx - x * tz,
        vz + w * tz + x * ty - y * tx,
    ], axis=1)

def get_quat(df, seg):
    return np.stack([
        df[f'SegmentQuaternions/Base/{seg}_W'].values,
        df[f'SegmentQuaternions/Base/{seg}_X'].values,
        df[f'SegmentQuaternions/Base/{seg}_Y'].values,
        df[f'SegmentQuaternions/Base/{seg}_Z'].values,
    ], axis=1)

def zy(arr): return -arr[:, 2], arr[:, 1]

datasets = [
    ('../data/processed/molab',              '../data/processed/molab_positions'),
    ('../data/processed/molab_cropped',      '../data/processed/molab_positions_cropped'),
    ('../data/processed/molab_cropped_down', '../data/processed/molab_positions_cropped_down'),
]

for in_dir, out_dir_pos in datasets:
    os.makedirs(out_dir_pos, exist_ok=True)

    for vid, fp, mid in mapping:
        height_cm, shoe_size, sex = participants[fp]
        height_m  = height_cm / 100
        ratios    = RATIOS[sex]
        thigh_len = height_m * ratios['thigh']
        shank_len = height_m * ratios['shank']
        foot_len  = EU_TO_FOOT_LENGTH_MM[shoe_size] / 1000

        df = pd.read_csv(f'{in_dir}/{vid}_molab.csv')
        n = len(df)

        l_shank_dir = rotate_vecs(get_quat(df, 'L_shank'), SHANK_LONG_AXIS)
        r_shank_dir = rotate_vecs(get_quat(df, 'R_shank'), SHANK_LONG_AXIS)
        l_thigh_dir = rotate_vecs(get_quat(df, 'L_thigh'), THIGH_LONG_AXIS)
        r_thigh_dir = rotate_vecs(get_quat(df, 'R_thigh'), THIGH_LONG_AXIS)
        l_foot_dir  = rotate_vecs(get_quat(df, 'L_foot'),  FOOT_LONG_AXIS)
        r_foot_dir  = rotate_vecs(get_quat(df, 'R_foot'),  FOOT_LONG_AXIS)

        l_shank_dir[:, 2] *= -1
        r_shank_dir[:, 2] *= -1
        l_thigh_dir[:, 2] *= -1
        r_thigh_dir[:, 2] *= -1
        l_foot_dir[:, 1] *= -1
        r_foot_dir[:, 1] *= -1

        l_ankle = np.zeros((n, 3))
        r_ankle = np.zeros((n, 3))
        l_toe   = l_ankle + l_foot_dir  * foot_len
        r_toe   = r_ankle + r_foot_dir  * foot_len
        l_knee  = l_ankle + l_shank_dir * shank_len
        r_knee  = r_ankle + r_shank_dir * shank_len
        l_hip   = l_knee  + l_thigh_dir * thigh_len
        r_hip   = r_knee  + r_thigh_dir * thigh_len

        l_toe_p   = zy(l_toe);   r_toe_p   = zy(r_toe)
        l_ankle_p = zy(l_ankle); r_ankle_p = zy(r_ankle)
        l_knee_p  = zy(l_knee);  r_knee_p  = zy(r_knee)
        l_hip_p   = zy(l_hip);   r_hip_p   = zy(r_hip)

        result = pd.DataFrame({
            'molab_frame':  df['molab_frame'],
            'molab_time':   df['molab_time'],
            'video_frame':  df['video_frame'],
            'video_time':   df['video_time'],
            'l_toe_x':      l_toe_p[0],    'l_toe_y':      l_toe_p[1],
            'l_ankle_x':    l_ankle_p[0],  'l_ankle_y':    l_ankle_p[1],
            'l_knee_x':     l_knee_p[0],   'l_knee_y':     l_knee_p[1],
            'l_hip_x':      l_hip_p[0],    'l_hip_y':      l_hip_p[1],
            'r_toe_x':      r_toe_p[0],    'r_toe_y':      r_toe_p[1],
            'r_ankle_x':    r_ankle_p[0],  'r_ankle_y':    r_ankle_p[1],
            'r_knee_x':     r_knee_p[0],   'r_knee_y':     r_knee_p[1],
            'r_hip_x':      r_hip_p[0],    'r_hip_y':      r_hip_p[1],
        })

        out_path = f'{out_dir_pos}/{vid}_positions.csv'
        result.to_csv(out_path, index=False)
        print(f'{vid} ({sex}, {height_cm}cm, EU{shoe_size}): {len(result)} rows -> {out_path}')

0028 (F, 167cm, EU39): 1798 rows -> ../data/processed/molab_positions/0028_positions.csv
0030 (F, 167cm, EU39): 4026 rows -> ../data/processed/molab_positions/0030_positions.csv
0045 (F, 168cm, EU38): 1990 rows -> ../data/processed/molab_positions/0045_positions.csv
0047 (F, 168cm, EU38): 3549 rows -> ../data/processed/molab_positions/0047_positions.csv
0059 (M, 178cm, EU43): 2657 rows -> ../data/processed/molab_positions/0059_positions.csv
0061 (M, 178cm, EU43): 3850 rows -> ../data/processed/molab_positions/0061_positions.csv
0074 (F, 168cm, EU38): 1843 rows -> ../data/processed/molab_positions/0074_positions.csv
0076 (F, 168cm, EU38): 3665 rows -> ../data/processed/molab_positions/0076_positions.csv
0091 (M, 169cm, EU41): 1466 rows -> ../data/processed/molab_positions/0091_positions.csv
0093 (M, 169cm, EU41): 3709 rows -> ../data/processed/molab_positions/0093_positions.csv
0028 (F, 167cm, EU39): 188 rows -> ../data/processed/molab_positions_cropped/0028_positions.csv
0030 (F, 167cm

### Save videos of cropped molab coordinates

In [9]:
import matplotlib.animation as animation
import os

STEP = 2
INTERVAL_MS = 20

for positions_dir, out_dir_anim, label in [
    ('../data/processed/molab_positions_cropped',      '../data/processed/molab_videos',      'sit-to-stand'),
    ('../data/processed/molab_positions_cropped_down', '../data/processed/molab_videos_down', 'stand-to-sit'),
]:
    os.makedirs(out_dir_anim, exist_ok=True)
    print(f'\n=== {label} ===')

    for vid, fp, mid in mapping:
        df = pd.read_csv(f'{positions_dir}/{vid}_positions.csv')
        df = df.iloc[::STEP].reset_index(drop=True)

        all_x = pd.concat([df['l_toe_x'], df['l_knee_x'], df['l_hip_x'],
                           df['r_toe_x'], df['r_knee_x'], df['r_hip_x']])
        all_y = pd.concat([df['l_toe_y'], df['l_knee_y'], df['l_hip_y'],
                           df['r_toe_y'], df['r_knee_y'], df['r_hip_y']])
        pad = 0.05
        xlim = (all_x.min() - pad, all_x.max() + pad)
        ylim = (all_y.min() - pad, all_y.max() + pad)

        fig, ax = plt.subplots(figsize=(4, 5))
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
        ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')

        line_r, = ax.plot([], [], '-o', color='tomato',    linewidth=2, markersize=5, label='right')
        line_l, = ax.plot([], [], '-o', color='steelblue', linewidth=2, markersize=5, label='left')
        title_obj = ax.set_title('')
        ax.legend(fontsize=8)
        plt.tight_layout()

        def make_funcs(df, vid, fp, line_l, line_r, title_obj):
            def init():
                line_l.set_data([], []); line_r.set_data([], [])
                title_obj.set_text('')
                return line_r, line_l, title_obj
            def update(i):
                row = df.iloc[i]
                for line, side in [(line_r, 'r'), (line_l, 'l')]:
                    xs = [row[f'{side}_toe_x'],   row[f'{side}_ankle_x'],
                          row[f'{side}_knee_x'],  row[f'{side}_hip_x']]
                    ys = [row[f'{side}_toe_y'],   row[f'{side}_ankle_y'],
                          row[f'{side}_knee_y'],  row[f'{side}_hip_y']]
                    line.set_data(xs, ys)
                title_obj.set_text(f'{vid}: {fp}  t={row["video_time"]:.2f}s')
                return line_r, line_l, title_obj
            return init, update

        init_fn, update_fn = make_funcs(df, vid, fp, line_l, line_r, title_obj)
        anim = animation.FuncAnimation(
            fig, update_fn, frames=len(df), init_func=init_fn,
            interval=INTERVAL_MS, blit=True
        )

        out_path = f'{out_dir_anim}/{vid}_molab_video.gif'
        anim.save(out_path, writer='pillow', fps=1000 // INTERVAL_MS)
        plt.close(fig)
        print(f'Saved {out_path}')


=== sit-to-stand ===
Saved ../data/processed/molab_videos/0028_molab_video.gif
Saved ../data/processed/molab_videos/0030_molab_video.gif
Saved ../data/processed/molab_videos/0045_molab_video.gif
Saved ../data/processed/molab_videos/0047_molab_video.gif
Saved ../data/processed/molab_videos/0059_molab_video.gif
Saved ../data/processed/molab_videos/0061_molab_video.gif
Saved ../data/processed/molab_videos/0074_molab_video.gif
Saved ../data/processed/molab_videos/0076_molab_video.gif
Saved ../data/processed/molab_videos/0091_molab_video.gif
Saved ../data/processed/molab_videos/0093_molab_video.gif

=== stand-to-sit ===
Saved ../data/processed/molab_videos_down/0028_molab_video.gif
Saved ../data/processed/molab_videos_down/0030_molab_video.gif
Saved ../data/processed/molab_videos_down/0045_molab_video.gif
Saved ../data/processed/molab_videos_down/0047_molab_video.gif
Saved ../data/processed/molab_videos_down/0059_molab_video.gif
Saved ../data/processed/molab_videos_down/0061_molab_video.gi